# Reverse RAG: Synthetic QA Dataset from Product Docs

In this tutorial, we create a synthetic evaluation dataset from product documentation.

In a standard RAG workflow, we start with a user question and retrieve relevant context. Here we reverse the process:
- start with product documentation
- build a small knowledge base
- retrieve source chunks
- extract facts from those chunks
- generate questions and reference answers from the facts

This gives us a grounded QA dataset for evaluating a support bot for the product.

As an optional final step, we ask a generic LLM to answer the generated questions without documentation context, then judge whether its answers are supported, incomplete, or contradictory.

## 1. Setup

We use:
- ChromaDB as a local vector store
- OpenAI embeddings through Chroma's embedding function
- OpenAI structured outputs to extract facts and generate QA pairs

No LangChain is used in this notebook.

In [1]:
 #!pip install openai chromadb pydantic requests

In [2]:
import os
import re
import uuid
from pathlib import Path
from typing import List, Literal

import chromadb
import pandas as pd
import requests
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from openai import OpenAI
from pydantic import BaseModel

pd.set_option("display.max_colwidth", None)

assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY before running this notebook."

client = OpenAI()

## 2. Load product documentation

We use Evidently documentation as a plain-text source.

The file is large, so we will later limit the number of chunks for a small tutorial run.

In [3]:
DOCS_URL = "https://docs.evidentlyai.com/llms-full.txt"

response = requests.get(DOCS_URL, timeout=60)
response.raise_for_status()

docs_text = response.text

print(f"Characters loaded: {len(docs_text):,}")
print(docs_text[:1000])

Characters loaded: 469,970
# Product updates
Source: https://docs.evidentlyai.com/changelog/changelog

Latest releases.

<Update label="2025-07-18" description="Evidently v0.7.11">
  ## **Evidently 0.7.11**

  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.11).

  Example notebooks:

  * Synthetic data generation: [code example](https://github.com/evidentlyai/evidently/blob/main/examples/cookbook/datagen.ipynb)
</Update>

<Update label="2025-07-09" description="Evidently v0.7.10">
  ## **Evidently 0.7.10**

  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.10).

  NEW: automated prompt optimization. Read the release blog on [prompt optimization for LLM judges](https://www.evidentlyai.com/blog/llm-judge-prompt-optimization).

  Example notebooks:

  * Code review binary LLM judge prompt optimization: [code example](https://github.com/evidentlyai/evidently/blob/main/examples/cookbook/prompt_optimization_

## 3. Split docs into chunks

We use a small plain-Python chunking function.

It splits text into overlapping word chunks. This is enough for the tutorial and avoids extra dependencies.

In [4]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_text(text: str, chunk_size: int = 900, overlap: int = 120) -> List[str]:
    words = clean_text(text).split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks


chunks = chunk_text(docs_text, chunk_size=900, overlap=120)

chunks_df = pd.DataFrame({
    "chunk_id": [f"chunk_{i:05d}" for i in range(len(chunks))],
    "chunk_index": range(len(chunks)),
    "text": chunks,
    "source_url": DOCS_URL,
})

print(f"Chunks created: {len(chunks_df)}")
chunks_df.head()

Chunks created: 66


chunk_id  chunk_index  \
0  chunk_00000            0   
1  chunk_00001            1   
2  chunk_00002            2   
3  chunk_00003            3   
4  chunk_00004            4   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

## 4. Select chunks for the tutorial run

For a production dataset, you can index the full documentation.

For this tutorial, we keep chunks likely to contain LLM evaluation and RAG-related content, then cap the size to control cost.

In [5]:
KEYWORDS = [
    "LLM",
    "evaluation",
    "eval",
    "descriptor",
    "dataset",
    "RAG",
    "retrieval",
    "tracing",
    "monitoring",
]

keyword_pattern = "|".join([re.escape(keyword) for keyword in KEYWORDS])

candidate_chunks_df = chunks_df[
    chunks_df["text"].str.contains(keyword_pattern, case=False, regex=True)
].copy()

MAX_CHUNKS_TO_INDEX = 250

candidate_chunks_df = (
    candidate_chunks_df
    .head(MAX_CHUNKS_TO_INDEX)
    .reset_index(drop=True)
)

print(f"Candidate chunks: {len(candidate_chunks_df)}")
candidate_chunks_df.head()

Candidate chunks: 66


chunk_id  chunk_index  \
0  chunk_00000            0   
1  chunk_00001            1   
2  chunk_00002            2   
3  chunk_00003            3   
4  chunk_00004            4   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

## 5. Build a ChromaDB knowledge base

Chroma can call the OpenAI embedding model for us.

This keeps the notebook simple: we add raw text chunks, and Chroma handles embeddings internally.

In [6]:
CHROMA_PATH = "./evidently_reverse_rag_chroma"
COLLECTION_NAME = f"evidently_docs_reverse_rag_{uuid.uuid4().hex[:8]}"

embedding_function = OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
)

collection.add(
    ids=candidate_chunks_df["chunk_id"].tolist(),
    documents=candidate_chunks_df["text"].tolist(),
    metadatas=candidate_chunks_df[["chunk_index", "source_url"]].to_dict(orient="records"),
)

collection.count()

66

## 6. Retrieve source chunks

Now we use the knowledge base to select chunks that can seed the synthetic dataset.

This is the reverse RAG step: retrieval is used to find source facts before questions exist.

In [7]:
SEED_QUERIES = [
    "How to evaluate LLM applications with Evidently",
    "LLM evaluation datasets descriptors and test suites",
    "RAG evaluation and retrieval quality",
    "Tracing and monitoring LLM applications",
]

def retrieve_chunks(query: str, n_results: int = 4) -> pd.DataFrame:
    result = collection.query(
        query_texts=[query],
        n_results=n_results,
    )

    return pd.DataFrame({
        "seed_query": query,
        "chunk_id": result["ids"][0],
        "chunk_text": result["documents"][0],
        "metadata": result["metadatas"][0],
        "distance": result["distances"][0],
    })


retrieved_chunks_df = pd.concat(
    [retrieve_chunks(query, n_results=4) for query in SEED_QUERIES],
    ignore_index=True,
)

retrieved_chunks_df = (
    retrieved_chunks_df
    .sort_values("distance")
    .drop_duplicates("chunk_id")
    .reset_index(drop=True)
)

retrieved_chunks_df.head()

seed_query     chunk_id  \
0  LLM evaluation datasets descriptors and test suites  chunk_00007   
1  LLM evaluation datasets descriptors and test suites  chunk_00029   
2                 RAG evaluation and retrieval quality  chunk_00030   
3                 RAG evaluation and retrieval quality  chunk_00065   
4  LLM evaluation datasets descriptors and test suites  chunk_00049   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## 7. Extract facts from chunks

For each retrieved chunk, we extract facts that are explicitly supported by the documentation.

These facts become the ground truth for synthetic QA generation.

In [8]:
class ExtractedFact(BaseModel):
    topic: str
    fact: str
    evidence_quote: str


class ExtractedFacts(BaseModel):
    facts: List[ExtractedFact]


FACT_EXTRACTION_INSTRUCTIONS = """Extract factual statements from the documentation chunk.

Rules:
- Use only facts explicitly supported by the chunk.
- Prefer facts useful for evaluating a support bot.
- Avoid marketing claims and vague statements.
- Keep each fact self-contained.
- Include a short evidence quote from the chunk.
"""


def extract_facts_from_chunk(chunk_text: str, max_facts: int = 3) -> ExtractedFacts:
    prompt = f"""Extract up to {max_facts} facts from this documentation chunk.

Documentation chunk:
{chunk_text}
"""

    response = client.responses.parse(
        model="gpt-4.1-mini",
        instructions=FACT_EXTRACTION_INSTRUCTIONS,
        input=prompt,
        text_format=ExtractedFacts,
        temperature=0,
    )

    return response.output_parsed


fact_rows = []

for _, row in retrieved_chunks_df.iterrows():
    extracted = extract_facts_from_chunk(row["chunk_text"], max_facts=3)

    for fact in extracted.facts:
        fact_rows.append({
            "chunk_id": row["chunk_id"],
            "chunk_text": row["chunk_text"],
            "metadata": row["metadata"],
            "topic": fact.topic,
            "source_fact": fact.fact,
            "evidence_quote": fact.evidence_quote,
        })

facts_df = pd.DataFrame(fact_rows)

facts_df.head()

chunk_id  \
0  chunk_00007   
1  chunk_00007   
2  chunk_00007   
3  chunk_00029   
4  chunk_00029   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

## 8. Generate questions and reference answers

Each question is generated from a documented fact.

The reference answer should be fully supported by the source fact and evidence quote.

In [9]:
class QAExample(BaseModel):
    question: str
    reference_answer: str


class QAExamples(BaseModel):
    examples: List[QAExample]


QA_GENERATION_INSTRUCTIONS = """Generate support-bot evaluation questions from documented facts.

Rules:
- The question should sound like a realistic user or developer asking about the product.
- The reference answer must be fully supported by the source fact.
- Do not add facts that are not present in the source fact or evidence.
- Keep the answer concise.
"""


def generate_qa_from_fact(source_fact: str, evidence_quote: str, n_questions: int = 1) -> QAExamples:
    prompt = f"""Generate {n_questions} QA example(s) from this documented fact.

Source fact:
{source_fact}

Evidence quote:
{evidence_quote}
"""

    response = client.responses.parse(
        model="gpt-4.1-mini",
        instructions=QA_GENERATION_INSTRUCTIONS,
        input=prompt,
        text_format=QAExamples,
        temperature=0.4,
    )

    return response.output_parsed


qa_rows = []

MAX_FACTS_FOR_QA = 10

for _, row in facts_df.head(MAX_FACTS_FOR_QA).iterrows():
    generated = generate_qa_from_fact(
        source_fact=row["source_fact"],
        evidence_quote=row["evidence_quote"],
        n_questions=1,
    )

    for example in generated.examples:
        qa_rows.append({
            "chunk_id": row["chunk_id"],
            "metadata": row["metadata"],
            "topic": row["topic"],
            "source_fact": row["source_fact"],
            "evidence_quote": row["evidence_quote"],
            "question": example.question,
            "reference_answer": example.reference_answer,
        })

qa_df = pd.DataFrame(qa_rows)

qa_df

,chunk_id,metadata,topic,source_fact,evidence_quote,question,reference_answer
0,chunk_00007,"{'chunk_index': 7, 'source_url': 'https://docs.evidentlyai.com/llms-full.txt'}",Dataset object creation,"To evaluate data with Evidently, you must create a Dataset object that attaches extra meta-information so data is processed correctly.","Once you have the data, you must create an Evidently `Dataset` object. This allows attaching extra meta-information so that your data is processed correctly.",How do I prepare my data for evaluation in Evidently?,"You need to create an Evidently Dataset object, which attaches extra meta-information to ensure your data is processed correctly."
1,chunk_00007,"{'chunk_index': 7, 'source_url': 'https://docs.evidentlyai.com/llms-full.txt'}",Use of two datasets,"Using two datasets (current and reference) allows side-by-side comparison, data drift detection, and simplifies test setup by generating test conditions automatically.",When to use two datasets: Side-by-side comparison... Data drift detection. (Required)... Simplify test setup. You can automatically generate test conditions... from the reference dataset.,"Why should I use two datasets, current and reference, in my tests?","Using two datasets allows side-by-side comparison, data drift detection, and simplifies test setup by automatically generating test conditions from the reference dataset."
2,chunk_00007,"{'chunk_index': 7, 'source_url': 'https://docs.evidentlyai.com/llms-full.txt'}",Descriptors in text evaluation,"Descriptors are row-level scores or labels assessing specific qualities of text data, used to evaluate LLM outputs and can be numerical, categorical, or text strings.","A Descriptor is a row-level score or label that assesses a specific quality of a given text... Each Descriptor returns a result that can be: Numerical, Categorical, Text string.",What types of results can Descriptors return when evaluating text data?,"Descriptors can return results that are numerical, categorical, or text strings."
3,chunk_00029,"{'chunk_index': 29, 'source_url': 'https://docs.evidentlyai.com/llms-full.txt'}",Email generation examples,"The documentation provides multiple examples of generated emails for various scenarios such as skipping meetings, sales pitches, outage notifications, polite follow-ups, and meeting reminders.",skip the meeting tomorrow — nothing new for me there... write a fluffy sales email that'll convert well... make it so they feel like they HAVE to reply... we have an outage idk when we resolve it... send a friendly meeting reminder...,Can the documentation help me write different types of emails like meeting reminders or sales pitches?,"Yes, the documentation provides multiple examples of generated emails for scenarios such as skipping meetings, sales pitches, outage notifications, polite follow-ups, and meeting reminders."
4,chunk_00029,"{'chunk_index': 29, 'source_url': 'https://docs.evidentlyai.com/llms-full.txt'}",Evaluation prompt for email appropriateness,A binary classification prompt template is defined to judge whether generated email texts are appropriate for U.S. corporate and workplace communication in tech companies.,You are an expert in U.S. corporate and workplace communication in tech companies... Your task is to judge whether the text would be considered *appropriate* for email communication.,What is the purpose of the binary classification prompt template described?,It is used to judge whether generated email texts are appropriate for U.S. corporate and workplace communication in tech companies.
5,chunk_00029,"{'chunk_index': 29, 'source_url': 'https://docs.evidentlyai.com/llms-full.txt'}",Multi-LLM evaluation setup,"The system uses multiple LLM evaluators (OpenAI GPT-4o-mini, Anthropic Claude-3.5, Gemini Gemini-2.0) to judge email appropriateness and aggregates their judgments including a descriptor to flag disagreements.",We create evaluators from multiple LLM providers... each judge includes a Pass condition..

## 9. Export the dataset

We save the synthetic QA dataset with traceable source facts.

This file can be used in later RAG or support-bot evaluation workflows.

In [10]:
output_path = "reverse_rag_synthetic_qa_dataset.csv"

qa_df.to_csv(output_path, index=False)

output_path

'reverse_rag_synthetic_qa_dataset.csv'

## 10. Takeaways

Reverse RAG turns documentation into an evaluation dataset.

Instead of starting with user questions, we start with facts that are known to exist in the product docs. Then we generate questions and reference answers from those facts.

This creates traceable evaluation examples: every question is linked back to a source chunk, fact, and evidence quote.

The same workflow can be extended with larger indexes, better seed queries, deduplication, and human review.